In [ ]:
%pip install scikit-learn nltk pandas


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully ✅")

Libraries loaded successfully ✅


In [6]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

print("Loading dataset...")
df = pd.read_csv("spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})
df['cleaned'] = df['message'].apply(preprocess_text)

print(f"Dataset loaded: {df.shape[0]} messages")
print(f"Spam: {(df['label'] == 'spam').sum()}")
print(f"Ham:  {(df['label'] == 'ham').sum()}")

Loading dataset...
Dataset loaded: 5572 messages
Spam: 747
Ham:  4825


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

y_pred = nb_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"ACCURACY: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

ACCURACY: 96.95%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       0.99      0.78      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [8]:
with open("nb_model.pkl", 'wb') as f:
    pickle.dump(nb_model, f)
print("nb_model.pkl saved! ✅")

with open("nb_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)
print("nb_vectorizer.pkl saved! ✅")

nb_model.pkl saved! ✅
nb_vectorizer.pkl saved! ✅
